# 01 - Build `ml.trip_validity_trips` and `ml.trip_validity_trip_fares`

Builds the base tables for the Trip Validity model, sourced directly from
`silver.afc_boardings` (not from the earlier `validation` schema, which
was exploratory only).

- `ml.trip_validity_trips`: one row per AFC trip (grouped by
  vehicle + trip_opened_at + trip_closed_at), scoped to November 2023.
- `ml.trip_validity_trip_fares`: one row per fare tap belonging to a trip
  in the table above, **including fares without GPS coordinates** (unlike
  the earlier `validation.trip_points`, which only kept geo-tagged fares).
  This is what lets every downstream notebook avoid ever touching `silver`
  again for trip/fare-level data.

In [1]:
import os
from pathlib import Path

import psycopg

In [2]:
# Root regardless of the kernel's actual cwd (nbconvert defaults it to the
# notebook's own directory, not the repo root), so opa_database.config's
# relative `.env` lookup resolves correctly either way.
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)

# Settings() requires raw_data_root to exist on disk (an ingest-pipeline
# concern) even though this notebook only needs db_dsn. Give it a
# placeholder that's guaranteed to exist, without touching .env or the
# shared config.py validator.
os.environ.setdefault("RAW_DATA_ROOT", str(_root))

'/home/victor/repos/opa-database'

In [3]:
from opa_database.config import settings

TRIP_DATE_START = "2023-11-01"
TRIP_DATE_END = "2023-12-01"

conn = psycopg.connect(settings.db_dsn)
conn.execute("CREATE SCHEMA IF NOT EXISTS ml;")
conn.commit()
print("ml schema ready")

ml schema ready


## Stage 1 - `ml.trip_validity_trips`

One row per trip, grouped from `silver.afc_boardings` by
`(service_date, vehicle_number, line_number, direction, trip_opened_at,
trip_closed_at)`. Verified earlier (against November 2023) that this key
alone is a clean 1:1 grouping — `line_number`/`direction`/`line_shift`
never vary within a `(vehicle_number, trip_opened_at, trip_closed_at)`
group, so including them in `GROUP BY` doesn't fragment trips further.

`bus_id`/`route_id` are zero-padded presentation forms of
`vehicle_number`/`line_number` (`lpad` only when *shorter* than the
target width, never truncating longer values — plain `lpad` in Postgres
truncates on overlong input, which would silently corrupt 4+ digit line
numbers).

`trip_duration_seconds` is `NULL`, not a garbage negative number, for the
handful of trips where `trip_closed_at` is the `1899-12-30` Delphi/OLE
zero-date sentinel (the AFC backend's way of saying "closing time was
never recorded") — confirmed earlier that this affects exactly 9 trips
in November 2023, all with `trip_closed_at < trip_opened_at`.

In [4]:
conn.execute("""
    DROP TABLE IF EXISTS ml.trip_validity_trips CASCADE;

    CREATE TABLE ml.trip_validity_trips (
        trip_id                 bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        bus_id                  text NOT NULL,
        route_id                text NOT NULL,
        route_direction         integer NOT NULL,
        trip_date               date NOT NULL,
        trip_hour                smallint NOT NULL,
        trip_opening_timestamp  timestamptz NOT NULL,
        trip_closing_timestamp  timestamptz NOT NULL,
        trip_duration_seconds   bigint,
        trip_fare_count         integer NOT NULL
    );
""")

conn.execute(
    """
    INSERT INTO ml.trip_validity_trips (
        bus_id, route_id, route_direction, trip_date, trip_hour,
        trip_opening_timestamp, trip_closing_timestamp,
        trip_duration_seconds, trip_fare_count
    )
    SELECT
        CASE WHEN length(vehicle_number) < 5
             THEN lpad(vehicle_number, 5, '0') ELSE vehicle_number END,
        CASE WHEN length(line_number) < 3
             THEN lpad(line_number, 3, '0') ELSE line_number END,
        direction,
        service_date,
        EXTRACT(hour FROM trip_opened_at AT TIME ZONE 'America/Fortaleza')::smallint,
        trip_opened_at,
        trip_closed_at,
        CASE WHEN trip_closed_at < trip_opened_at THEN NULL
             ELSE EXTRACT(EPOCH FROM (trip_closed_at - trip_opened_at))::bigint
        END,
        count(*)
    FROM silver.afc_boardings
    WHERE service_date >= %(start)s AND service_date < %(end)s
    GROUP BY service_date, vehicle_number, line_number, direction,
             trip_opened_at, trip_closed_at;
    """,
    {"start": TRIP_DATE_START, "end": TRIP_DATE_END},
)

conn.execute("""
    CREATE INDEX trip_validity_trips_bus_id_idx
        ON ml.trip_validity_trips (bus_id);
    CREATE INDEX trip_validity_trips_route_id_idx
        ON ml.trip_validity_trips (route_id);
    CREATE INDEX trip_validity_trips_trip_date_idx
        ON ml.trip_validity_trips (trip_date);
    CREATE INDEX trip_validity_trips_trip_hour_idx
        ON ml.trip_validity_trips (trip_hour);
    ANALYZE ml.trip_validity_trips;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.trip_validity_trips;")
    print("trip_validity_trips rows:", cur.fetchone()[0])

trip_validity_trips rows: 940988


## Stage 2 - `ml.trip_validity_trip_fares`

One row per fare tap (**all of them**, not just geo-tagged) belonging to
a trip above, joined back to `silver.afc_boardings` by the natural key
`(service_date, padded vehicle_number, trip_opened_at, trip_closed_at)` —
the only reliable rejoin path, since `trip_id` here is a plain sequential
identity with no formula back to the source.

`event_id` gets a **partial** unique index (`WHERE event_id <> '0'`),
mirroring `silver.afc_boardings`'s own unique index: `'0'` is a sentinel
for "no event id assigned" and is not actually unique in the source data
(confirmed: 211,044 such rows in November 2023 among just the geo-tagged
subset alone). `fare_id` is the real surrogate primary key.

`latitude`/`longitude`/`geom` are nullable — `geom` is a `STORED`
generated column exactly like `silver.afc_boardings`'s own, and
`ST_MakePoint`/`ST_SetSRID` are strict functions that already return
`NULL` on `NULL` input, so no `CASE` is needed for the ungeotagged rows.

In [5]:
conn.execute("""
    DROP TABLE IF EXISTS ml.trip_validity_trip_fares CASCADE;

    CREATE TABLE ml.trip_validity_trip_fares (
        fare_id      bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        event_id     text NOT NULL,
        trip_id      bigint NOT NULL REFERENCES ml.trip_validity_trips (trip_id),
        boarding_at  timestamptz NOT NULL,
        latitude     double precision,
        longitude    double precision,
        geom         geometry(Point, 4326) GENERATED ALWAYS AS (
                         ST_SetSRID(ST_MakePoint(longitude, latitude), 4326)
                     ) STORED
    );
""")

conn.execute(
    """
    INSERT INTO ml.trip_validity_trip_fares (
        event_id, trip_id, boarding_at, latitude, longitude
    )
    SELECT
        b.event_id,
        t.trip_id,
        b.boarding_at,
        b.latitude,
        b.longitude
    FROM silver.afc_boardings b
    JOIN ml.trip_validity_trips t
      ON t.trip_date = b.service_date
     AND t.bus_id = CASE WHEN length(b.vehicle_number) < 5
                          THEN lpad(b.vehicle_number, 5, '0') ELSE b.vehicle_number END
     AND t.trip_opening_timestamp = b.trip_opened_at
     AND t.trip_closing_timestamp = b.trip_closed_at
    WHERE b.service_date >= %(start)s AND b.service_date < %(end)s;
    """,
    {"start": TRIP_DATE_START, "end": TRIP_DATE_END},
)

conn.execute(
    "CREATE UNIQUE INDEX trip_validity_trip_fares_event_id_key "
    "ON ml.trip_validity_trip_fares (event_id) WHERE event_id <> '0';"
)
conn.execute("""
    CREATE INDEX trip_validity_trip_fares_trip_id_idx
        ON ml.trip_validity_trip_fares (trip_id);
    CREATE INDEX trip_validity_trip_fares_geom_idx
        ON ml.trip_validity_trip_fares USING GIST (geom);
    CREATE INDEX trip_validity_trip_fares_boarding_at_idx
        ON ml.trip_validity_trip_fares (boarding_at);
    ANALYZE ml.trip_validity_trip_fares;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.trip_validity_trip_fares;")
    print("trip_validity_trip_fares rows:", cur.fetchone()[0])
    cur.execute("""
        SELECT count(*) FROM ml.trip_validity_trip_fares WHERE geom IS NOT NULL;
    """)
    print("  of which geo-tagged:", cur.fetchone()[0])

trip_validity_trip_fares rows: 15381356


  of which geo-tagged: 12769470


## Stage 3 - backfill `trip_path` and `trip_fares_with_latlon` onto `trips`

These depend on `trip_fares` existing, so they're added after the fact
rather than in the Stage 1 `CREATE TABLE`.

- `trip_fares_with_latlon`: count of this trip's fares that have a
  coordinate (`0` default, not `NULL`, for trips with none at all).
- `trip_path`: `ST_MakeLine(geom ORDER BY boarding_at)` over only the
  geo-tagged fares — one row per trip, no row explosion. `NULL` when a
  trip has 0 or 1 geo-tagged fares (a `LineString` needs ≥ 2 points;
  confirmed this happens for real single-point trips, e.g. a trip with
  exactly one geo-tagged fare tap).

In [6]:
conn.execute("""
    ALTER TABLE ml.trip_validity_trips
        ADD COLUMN trip_fares_with_latlon integer NOT NULL DEFAULT 0;
    ALTER TABLE ml.trip_validity_trips
        ADD COLUMN trip_path geometry(LineString, 4326);
""")

conn.execute("""
    UPDATE ml.trip_validity_trips t
    SET trip_fares_with_latlon = f.n
    FROM (
        SELECT trip_id, count(*) AS n
        FROM ml.trip_validity_trip_fares
        WHERE geom IS NOT NULL
        GROUP BY trip_id
    ) f
    WHERE f.trip_id = t.trip_id;
""")

conn.execute("""
    UPDATE ml.trip_validity_trips t
    SET trip_path = p.path
    FROM (
        SELECT trip_id, ST_MakeLine(geom ORDER BY boarding_at) AS path
        FROM ml.trip_validity_trip_fares
        WHERE geom IS NOT NULL
        GROUP BY trip_id
    ) p
    WHERE p.trip_id = t.trip_id;
""")

conn.execute("""
    CREATE INDEX trip_validity_trips_trip_path_gix
        ON ml.trip_validity_trips USING GIST (trip_path);
""")
conn.execute("ANALYZE ml.trip_validity_trips;")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT sum(trip_fares_with_latlon),
               count(*) FILTER (WHERE trip_path IS NOT NULL)
        FROM ml.trip_validity_trips;
    """)
    latlon_sum, paths = cur.fetchone()
    print("sum(trip_fares_with_latlon):", latlon_sum)
    print("trips with a non-null trip_path:", paths)

sum(trip_fares_with_latlon): 12769470
trips with a non-null trip_path: 899299


## Column-level provenance comments

Documented on the tables themselves (`COMMENT ON COLUMN`, queryable via
`\d+` or `information_schema.columns` forever, independent of this
notebook) as well as here in markdown.

In [7]:
from psycopg import sql

TRIPS_COMMENTS = {
    "trip_id": (
        "Surrogate identity PK. No formula back to source; rejoin via the "
        "natural key (trip_date, bus_id, trip_opening_timestamp, "
        "trip_closing_timestamp) against silver.afc_boardings."
    ),
    "bus_id": (
        "silver.afc_boardings.vehicle_number, zero-padded to 5 chars "
        "(never truncated if already longer)."
    ),
    "route_id": (
        "silver.afc_boardings.line_number, zero-padded to a minimum of 3 "
        "chars (never truncated if already longer)."
    ),
    "route_direction": (
        "silver.afc_boardings.direction, as-is (0/1, AFC's own 'sentido')."
    ),
    "trip_date": "silver.afc_boardings.service_date, as-is.",
    "trip_hour": (
        "Hour (0-23) of trip_opening_timestamp converted to "
        "America/Fortaleza local time."
    ),
    "trip_opening_timestamp": "silver.afc_boardings.trip_opened_at, as-is (UTC).",
    "trip_closing_timestamp": "silver.afc_boardings.trip_closed_at, as-is (UTC).",
    "trip_duration_seconds": (
        "trip_closing_timestamp - trip_opening_timestamp in seconds. NULL "
        "when trip_closing_timestamp < trip_opening_timestamp (the "
        "1899-12-30 Delphi zero-date sentinel meaning closing time was "
        "never recorded)."
    ),
    "trip_fare_count": (
        "count(*) of all fare taps (rows) on this trip from "
        "silver.afc_boardings, geo-tagged or not."
    ),
    "trip_fares_with_latlon": (
        "count of this trip's rows in trip_fares with a non-null geom. 0 "
        "(not NULL) when none."
    ),
    "trip_path": (
        "ST_MakeLine(geom ORDER BY boarding_at) over this trip's "
        "geo-tagged fares in trip_fares. NULL when fewer than 2 "
        "geo-tagged fares exist."
    ),
}

FARES_COMMENTS = {
    "fare_id": "Surrogate identity PK.",
    "event_id": (
        "silver.afc_boardings.event_id, as-is. Not globally unique here: "
        "'0' is a sentinel for 'no event id assigned' shared by many "
        "rows, so the unique index on this column excludes event_id = '0'."
    ),
    "trip_id": "FK to ml.trip_validity_trips.trip_id.",
    "boarding_at": "silver.afc_boardings.boarding_at, as-is (UTC).",
    "latitude": (
        "silver.afc_boardings.latitude, as-is. NULL when the validator "
        "had no GPS fix at tap time."
    ),
    "longitude": (
        "silver.afc_boardings.longitude, as-is. NULL when the validator "
        "had no GPS fix at tap time."
    ),
    "geom": (
        "Generated: ST_SetSRID(ST_MakePoint(longitude, latitude), 4326). "
        "NULL whenever latitude/longitude are NULL."
    ),
}


def comment_on_column(cur: psycopg.Cursor, table: str, col: str, text: str) -> None:
    """Apply a COMMENT ON COLUMN for one column via safe SQL composition."""
    cur.execute(
        sql.SQL("COMMENT ON COLUMN ml.{}.{} IS {};").format(
            sql.Identifier(table), sql.Identifier(col), sql.Literal(text)
        )
    )


with conn.cursor() as cur:
    for col, text in TRIPS_COMMENTS.items():
        comment_on_column(cur, "trip_validity_trips", col, text)
    for col, text in FARES_COMMENTS.items():
        comment_on_column(cur, "trip_validity_trip_fares", col, text)

conn.execute(
    sql.SQL("COMMENT ON TABLE ml.trip_validity_trips IS {};").format(
        sql.Literal(
            "Trip Validity model: one row per AFC trip, November 2023, built directly "
            "from silver.afc_boardings. See "
            "ml/trip_validity_model/notebooks/01_build_trip_tables.ipynb."
        )
    )
)
conn.execute(
    sql.SQL("COMMENT ON TABLE ml.trip_validity_trip_fares IS {};").format(
        sql.Literal(
            "Trip Validity model: one row per fare tap (all of them, not just "
            "geo-tagged) on a trip in trip_validity_trips. See "
            "ml/trip_validity_model/notebooks/01_build_trip_tables.ipynb."
        )
    )
)
conn.commit()
print("comments applied")

comments applied


## Verification

In [8]:
import polars as pl

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            (SELECT count(*) FROM ml.trip_validity_trips) AS trips,
            (SELECT count(*) FROM ml.trip_validity_trip_fares) AS fares,
            (SELECT count(*) FROM ml.trip_validity_trip_fares
                WHERE geom IS NOT NULL) AS geo_fares,
            (SELECT count(*) FROM ml.trip_validity_trips
                WHERE trip_duration_seconds IS NULL) AS null_duration_trips,
            (SELECT sum(trip_fare_count) FROM ml.trip_validity_trips)
                AS sum_trip_fare_count;
    """)
    cols = [d.name for d in cur.description]
    row = cur.fetchone()

summary = pl.DataFrame([dict(zip(cols, row, strict=True))])
print(summary)

# sanity: sum(trip_fare_count) across trips must equal total fare rows,
# and must equal the source row count for November 2023 (15,381,356, per
# the earlier validation.trips build).
if summary["fares"][0] != summary["sum_trip_fare_count"][0]:
    msg = "fare_count mismatch"
    raise AssertionError(msg)
print("OK: trip_fare_count sums match trip_fares row count")

shape: (1, 5)
┌────────┬──────────┬───────────┬─────────────────────┬─────────────────────┐
│ trips  ┆ fares    ┆ geo_fares ┆ null_duration_trips ┆ sum_trip_fare_count │
│ ---    ┆ ---      ┆ ---       ┆ ---                 ┆ ---                 │
│ i64    ┆ i64      ┆ i64       ┆ i64                 ┆ i64                 │
╞════════╪══════════╪═══════════╪═════════════════════╪═════════════════════╡
│ 940988 ┆ 15381356 ┆ 12769470  ┆ 9                   ┆ 15381356            │
└────────┴──────────┴───────────┴─────────────────────┴─────────────────────┘
OK: trip_fare_count sums match trip_fares row count
